### Importing Libraries and Initilize LLM Client

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

llm = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
model_name = "minimax-m2.5:cloud"

### Defining Tool (Function)

In [3]:
from datetime import datetime

def get_current_time():
    """
    Returns the current time as a string.
    """
    return datetime.now().strftime("%H:%M:%S")

In [ ]:
get_current_time()

In [ ]:
prompt = "What is the current time?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

response = llm.chat.completions.create(
    model=model_name, 
    messages=messages,
    )
resp = response.choices[0].message.content
print(resp)

#### LLM with Tool call

In [5]:
prompt = "What is the current time?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

In [6]:
tools = [
       {
           "type": "function",
           "function": {
               "name": "get_current_time",
               "description": "Returns the current time as a string.",
               "parameters": {
                   "type": "object",
                   "properties": {},
                   "required": [],
               },
           },
       }
   ]

In [ ]:
# model_name = "gpt-4.1-nano"
response = llm.chat.completions.create(
    model=model_name, 
    messages=messages,
    tools= tools
    )
msg = response.choices[0].message
print(msg)

In [ ]:
if msg.tool_calls:

    tool_name = msg.tool_calls[0].function.name

    if tool_name == "get_current_time":
        result = get_current_time()

        messages.append(msg)

        messages.append({
            "role": "tool",
            "tool_call_id": msg.tool_calls[0].id,
            "content": result
        })

        final = llm.chat.completions.create(
            model=model_name,
            messages=messages
        )

        print(final.choices[0].message.content)

In [9]:
from zoneinfo import ZoneInfo

def get_time_from_zone(zone):
    """
    Returns the current time in a specific timezone.
    """
    return datetime.now(ZoneInfo(zone)).strftime("%H:%M:%S")

In [10]:
arg_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_time_from_zone",
            "description": "Returns the current time in a specific timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "zone": {
                        "type": "string",
                        "description": "IANA timezone name, e.g. 'Asia/Kolkata' or 'America/New_York'."
                    }
                },
                "required": ["zone"]
            },
        },
    }
]

In [ ]:
prompt = "What is the current time in London?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

response = llm.chat.completions.create(
    model=model_name, 
    messages=messages,
    tools= arg_tools
    )
msg = response.choices[0].message
print(msg)

In [12]:
import json
if msg.tool_calls:

    tool_name = msg.tool_calls[0].function.name
    arg = msg.tool_calls[0].function.arguments
    zone = json.loads(arg)['zone']
    print(json.loads(arg)['zone'])

    if tool_name == "get_time_from_zone":
        result = get_time_from_zone(zone)

        messages.append(msg)

        messages.append({
            "role": "tool",
            "tool_call_id": msg.tool_calls[0].id,
            "content": result
        })

        final = llm.chat.completions.create(
            model=model_name,
            messages=messages
        )

        print(final.choices[0].message.content)

Europe/London
The current time in London is **07:24:18**.


### Agent using OpenAI SDK

In [ ]:
! pip install openai-agents

In [5]:
from agents import Agent, Runner, set_tracing_disabled

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')

client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

llm = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
model_name = "minimax-m2.5:cloud"

In [1]:
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, function_tool


client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# client = AsyncOpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=google_api_key)


model = OpenAIChatCompletionsModel(
    model="minimax-m2.5:cloud",
    openai_client=client
)


In [2]:
from datetime import datetime

@function_tool
def get_current_time():
    """
    Returns the current time as a string.
    """
    return datetime.now().strftime("%H:%M:%S")


from zoneinfo import ZoneInfo

@function_tool
def get_time_from_zone(zone):
    """
    Returns the current time in a specific timezone.
    """
    return datetime.now(ZoneInfo(zone)).strftime("%H:%M:%S")

In [ ]:
get_time_from_zone

In [10]:
set_tracing_disabled(disabled=True)


instructions ="You are a helpful assistant that can answer questions about current time. Use tools when required."
tools = [get_current_time, get_time_from_zone]
agent = Agent(
    name = "Date Agent",
    instructions = instructions,
    model=model,
    tools=tools,
)



In [ ]:
result = await Runner.run(agent, "What is the current time in New Delhi?")
print(result.final_output)

### Adding Function as Tool to Improve Knowledge Base

In [ ]:
result = await Runner.run(agent, "Can you tell about Pratik Joshi?")
print(result.final_output)

In [3]:
@function_tool
def get_pratik_joshi_details():
    """
    Returns all the details about Pratik Joshi.
    """
    summary = "Pratik Joshi is an Indian Cricket. He is opener batsman and bowler. He plays for RCB in IPL"
    return summary

In [6]:
instructions ="You are a helpful assistant that can answer user question. Use tools when required."
tools = [get_current_time, get_time_from_zone, get_pratik_joshi_details]
fake_agent = Agent(
    name = "Date Agent",
    instructions = instructions,
    model=model,
    tools=tools,
)

In [ ]:
set_tracing_disabled(disabled=True)

result = await Runner.run(fake_agent, "Can you tell about Pratik Joshi?")
print(result.final_output)